# Entities

## Some definitions

- a `mammos_units.Quantity` is an object that carries a value and a unit.

- a `mammos_entity.Entity` links a quantity to its definition in the [EMMO ontology](https://github.com/emmo-repo/EMMO) and the [MaMMoS additions for magnetic materials](https://github.com/MaMMoS-project/MagneticMaterialsOntology)


In [1]:
import mammos_units as u

import mammos_entity as me

## Creating entities

For entities that are important in MaMMoS, there are [convenient attributes](https://mammos-project.github.io/mammos/api/mammos_entity.html) to define those. For example for the {entity}`SpontaneousMagnetization`:

In [2]:
Ms = me.Ms(800e3, "A/m")  # defines Ms = 8e5 A/m
Ms

Entity(ontology_label='SpontaneousMagnetization', value=np.float64(800000.0), unit='A / m')

If no units are provided, then default units are inferred from the ontology (`A/m` in the example above). These are typically SI units without numerical prefactors (such as kilo, milli, etc.). Inferring units from the ontology is fragile and the inferred units can change between different releases of `mammos-entity` without notice. Therefore, it is highly recommended to always pass units explicitly.

If units are provided, these are compared with the units expected for that entity. An error is raised if they do not match.

In [3]:
M1 = me.Ms(800e3, "A/m")
M1

Entity(ontology_label='SpontaneousMagnetization', value=np.float64(800000.0), unit='A / m')

Providing units can be also useful, if numerical prefactors are used (such as kilo):

In [4]:
M2 = me.Ms(800, "kA/m")
M2

Entity(ontology_label='SpontaneousMagnetization', value=np.float64(800.0), unit='kA / m')

In [5]:
M1 == M2

True

We can create an entity from a Quantity as well:

In [6]:
Ms_quantity = 800 * u.kA / u.m  # or equivalently: Ms_quantity = u.Quantity(800, "kA/m")
Ms_quantity

<Quantity 800. kA / m>

In [7]:
me.Ms(Ms_quantity)

Entity(ontology_label='SpontaneousMagnetization', value=np.float64(800.0), unit='kA / m')

Entities can also have an optional attribute, `description`, storing a string. This description will appear in the string representation:

In [8]:
me.Ms(Ms_quantity, description="Evaluated using UppASD with 70000 Monte Carlo steps.")

Entity(ontology_label='SpontaneousMagnetization', value=np.float64(800.0), unit='kA / m', description='Evaluated using UppASD with 70000 Monte Carlo steps.')

By default, entities are displayed with their HTML representation in notebooks. To get a plain-text representation use one of:

In [9]:
print(me.Ms(Ms_quantity, description="Evaluated using UppASD with 70000 Monte Carlo steps."))

SpontaneousMagnetization(value=800.0, unit=kA / m, description='Evaluated using UppASD with 70000 Monte Carlo steps.')


In [10]:
repr(me.Ms(Ms_quantity, description="Evaluated using UppASD with 70000 Monte Carlo steps."))

"Entity(ontology_label='SpontaneousMagnetization', value=np.float64(800.0), unit='kA / m', description='Evaluated using UppASD with 70000 Monte Carlo steps.')"

If we like to express spontaneous magnetization in Tesla, we can convert it to the required A/m. To allow that conversion we must explicitely enable the `magnetic_flux_field` equivalency ($B=\mu_0H$):

In [11]:
# enable implicit conversion between A/m and T in the rest of the notebook:
u.set_enabled_equivalencies(u.magnetic_flux_field())

a = 1.2 * u.T  # Quantity in Tesla

Ms = me.Ms(a.to("A/m"))  # Convert Quantity to A/m and create entity
Ms

Entity(ontology_label='SpontaneousMagnetization', value=np.float64(954929.6580315315), unit='A / m')

As a cross-check we can access the quantity attribute and convert that back to Tesla. This operation does not modify `Ms`.

In [12]:
Ms.q.to("T")

<Quantity 1.2 T>

Initializing with incompatible units fails with an error. In particular, implicit conversion via equivalencies is not supported. Convert to the required units first, as shown above.

In [13]:
me.Ms(1.2, "T")  # Tesla not compatible with A/m

ValueError: Given unit: T incompatible with ontology. Allowed units for entity SpontaneousMagnetization are: (Unit("A / m"),).

For more details about unit conversion and equivalencies refer to the documentation of [mammos-units](https://mammos-project.github.io/mammos/examples/mammos-units/example.html#Equivalency).

## Access to ontology

Each entity object knows its label in the ontology:

In [14]:
Ms.ontology_label

'SpontaneousMagnetization'

and its unique identifier of the ontology entry, the so-called IRI (Internationalized Resource Identifier): 

In [15]:
Ms.ontology_iri

'https://w3id.org/emmo/domain/magnetic-materials#EMMO_032731f8-874d-5efb-9c9d-6dafaa17ef25'

When saving data to a file, the property `ontology_label_with_iri` might be useful to save in the metadata as it returns a string containing the ontology label together with the IRI.

In [16]:
Ms.ontology_label_with_iri

'SpontaneousMagnetization https://w3id.org/emmo/domain/magnetic-materials#EMMO_032731f8-874d-5efb-9c9d-6dafaa17ef25'

The ontology attribute links to an [owlready2](https://github.com/pwin/owlready2) object created by the [EMMOntoPy](https://emmo-repo.github.io/EMMOntoPy/stable/) package. We can use all attributes of the ontology object through `Ms.ontology`:

In [17]:
Ms.ontology.get_annotations()

{'prefLabel': [locstr('SpontaneousMagnetization', 'en-US')],
 'elucidation': [locstr('The spontaneous magnetization, Ms, of a ferromagnet is the result\nof alignment of the magnetic moments of individual atoms. Ms exists\nwithin a domain of a ferromagnet.', 'en')],
 'altLabel': [locstr('Ms', ''), locstr('SpontaneousMagnetisation', 'en-GB')],
 'wikipediaReference': [locstr('https://en.wikipedia.org/wiki/Spontaneous_magnetization', '')],
 'IECEntry': [locstr('https://www.electropedia.org/iev/iev.nsf/display?openform&ievref=221-02-41', '')]}

In [18]:
Ms.ontology.get_class_properties()

{core.altLabel,
 emmo.elucidation,
 emmo.hasMeasurementUnit,
 magnetic-materials.wikipediaReference,
 core.prefLabel,
 magnetic-materials.IECEntry}

In [19]:
print(Ms.ontology.elucidation[0])

The spontaneous magnetization, Ms, of a ferromagnet is the result
of alignment of the magnetic moments of individual atoms. Ms exists
within a domain of a ferromagnet.


## Entities contain Quantities

Each entity has a `quantity` attribute containing the {class}`mammos_units.Quantity` object. See [mammos_units examples](https://mammos-project.github.io/mammos/examples/mammos-units/index.html) for details.

In [20]:
Ms.quantity

<Quantity 954929.65803153 A / m>

There is also the shorthand `q` for the `quantity` attribute:

In [21]:
Ms.q

<Quantity 954929.65803153 A / m>

The attributes `value` and `unit` provide direct access to value and unit of the quantity:

In [22]:
Ms.value

np.float64(954929.6580315315)

In [23]:
Ms.unit

Unit("A / m")

Entities do not support numerical operations. To perform these operations, we first need to extract the quantity:

In [24]:
Ms.q**2

<Quantity 9.11890652e+11 A2 / m2>

In [25]:
Ms.quantity.to("kA/m")

<Quantity 954.92965803 kA / m>

In [26]:
Ms.quantity.to("T")

<Quantity 1.2 T>

## Defining vector entities (Example Zeeman field)

In [27]:
H = me.H([1e4, 1e4, 1e4], "A/m")
H

Entity(ontology_label='ExternalMagneticField', value=array([10000., 10000., 10000.]), unit='A / m')

In [28]:
H.ontology

magnetic-materials.ExternalMagneticField

In [29]:
print(H.ontology.elucidation[0])

The external field H′, acting on a sample that is produced by
electric currents or the stray field of magnets outside the sample
volume, is often called the applied field.


## Slicing Entities

Entities support indexing and slicing operations, just like NumPy arrays. All indexing operations supported by NumPy are valid, including integers, slices, boolean arrays, and integer arrays.

In [30]:
# Create an entity with multiple values
Ms_data = me.Ms([500, 600, 700, 800], "kA/m", description="Temperature-dependent measurements")
Ms_data

Entity(ontology_label='SpontaneousMagnetization', value=array([500., 600., 700., 800.]), unit='kA / m', description='Temperature-dependent measurements')

Integer indexing returns a scalar entity:

In [31]:
Ms_data[0]

Entity(ontology_label='SpontaneousMagnetization', value=np.float64(500.0), unit='kA / m', description='Temperature-dependent measurements')

Slice indexing returns an entity with a subset of values:

In [32]:
Ms_data[1:3]

Entity(ontology_label='SpontaneousMagnetization', value=array([600., 700.]), unit='kA / m', description='Temperature-dependent measurements')

Boolean indexing selects values where the mask is `True`:

In [33]:
Ms_data[[True, False, True, False]]

Entity(ontology_label='SpontaneousMagnetization', value=array([500., 700.]), unit='kA / m', description='Temperature-dependent measurements')

Integer array indexing selects values at given positions:

In [34]:
Ms_data[[0, 2, 3]]

Entity(ontology_label='SpontaneousMagnetization', value=array([500., 700., 800.]), unit='kA / m', description='Temperature-dependent measurements')

Slicing with `[::-1]` reverses the order of values:

In [35]:
Ms_data[::-1]

Entity(ontology_label='SpontaneousMagnetization', value=array([800., 700., 600., 500.]), unit='kA / m', description='Temperature-dependent measurements')

Integer array indexing can reorder elements:

In [36]:
Ms_data[[3, 0, 1, 2]]

Entity(ontology_label='SpontaneousMagnetization', value=array([800., 500., 600., 700.]), unit='kA / m', description='Temperature-dependent measurements')

The ontology label, unit, and description are preserved after slicing:

In [37]:
sliced = Ms_data[1:]
print(f"Ontology label: {sliced.ontology_label}")
print(f"Unit: {sliced.unit}")
print(f"Description: {sliced.description}")

Ontology label: SpontaneousMagnetization
Unit: kA / m
Description: Temperature-dependent measurements


## Extending an `Entity`

Entities are immutable and adding additional values is not directly possible. Instead a new entity with the combined values has to be created. This can be conveniently done with the following function:

In [38]:
T1 = me.T([1, 2], "K")
T2 = me.T([3, 4], "K")

me.operations.concat_flat(T1, T2)

Entity(ontology_label='ThermodynamicTemperature', value=array([1., 2., 3., 4.]), unit='K')

`concat_flat` is not limited to entities. You can also pass {class}`mammos_units.Quantity` or other scalar or array-like data. `concat_flat` will check that all elements are compatible (same ontology_label if they are entities, compatible units for entities and quantities) and convert everything into a single entity:

In [39]:
me.operations.concat_flat(me.T([1, 2]), 3 * u.K, [4, 5])

Entity(ontology_label='ThermodynamicTemperature', value=array([1., 2., 3., 4., 5.]), unit='K')

## Does `mammos_entity` not provide your preferred entity?

The list of convenience attributes is at 
https://mammos-project.github.io/mammos/api/mammos_entity.html

If the desired entity is not available, we can search the EMMO ontology (including the `magnetic_material_mammos` additions), for example for entity labels containing the string "Magnetization":

In [40]:
me.search_labels("Magnetization")

['MagneticMomentPerUnitMass',
 'Magnetization',
 'MassMagnetizationUnit',
 'Remanence',
 'SaturationMagnetization',
 'SpontaneousMagnetization']

The function searches for the given substring in all labels (prefLabel, altLabel). If any label contains the substring the prefLabel is returned. Therefore, we see "MagneticMomentPerUnitMass" in the example above.

We can also search for exact matches only:

In [41]:
me.search_labels("Magnetization", auto_wildcard=False)

['Magnetization']

The function `search_labels` is a wrapper around [`get_by_label_all` from emmontopy](https://emmo-repo.github.io/EMMOntoPy/latest/api_reference/ontopy/ontology/#ontopy.ontology.Ontology.get_by_label_all). If you need any of the additional options use the function directly. Note that the return type of the elements changes from strings (prefLabel) to `owlready2.Thing` classes.

In [42]:
me.mammos_ontology.get_by_label_all("*Field*")

{emmo.ElectricDisplacementFieldUnit,
 emmo.ElectricFieldStrength,
 emmo.ElectricFieldStrengthUnit,
 emmo.MagneticFieldStrength,
 emmo.MagneticFieldStrengthUnit,
 magnetic-materials.AnisotropyField,
 magnetic-materials.CoercivityHc,
 magnetic-materials.DemagnetizingField,
 magnetic-materials.ExternalMagneticField,
 magnetic-materials.InternalMagneticField,
 magnetic-materials.KneeField,
 magnetic-materials.KneeFieldExternal,
 magnetic-materials.MokeAppliedField,
 magnetic-materials.SwitchingFieldCoercivity,
 magnetic-materials.SwitchingFieldCoercivityExternal}

In [43]:
me.mammos_ontology.get_by_label_all("*Field*", prefix="magnetic-materials")

{magnetic-materials.AnisotropyField,
 magnetic-materials.CoercivityHc,
 magnetic-materials.DemagnetizingField,
 magnetic-materials.ExternalMagneticField,
 magnetic-materials.InternalMagneticField,
 magnetic-materials.KneeField,
 magnetic-materials.KneeFieldExternal,
 magnetic-materials.MokeAppliedField,
 magnetic-materials.SwitchingFieldCoercivity,
 magnetic-materials.SwitchingFieldCoercivityExternal}

Once identified the right label, we create an entity like this:

In [44]:
me.Entity("AnisotropyField", value=230, unit="A/m")

Entity(ontology_label='AnisotropyField', value=np.float64(230.0), unit='A / m')

To search in fields other than labels, e.g. in all elucidations, use the following function:

In [45]:
me.mammos_ontology.search(elucidation="*magnetization*")

[emmo.AmpereTurnPerMetre, emmo.Permeability, emmo.AmperePerMetre, emmo.MagneticPolarisation, emmo.Coercivity, magnetic-materials.ExternalSusceptibility, magnetic-materials.KneeFieldExternal, magnetic-materials.DemagnetizingFactor, magnetic-materials.KneeField, magnetic-materials.UniaxialMagnetocrystallineAnisotropy, magnetic-materials.ShapeAnisotropyConstant, magnetic-materials.UniaxialMagneticAnisotropy, magnetic-materials.MagneticHysteresisProperties, magnetic-materials.CoercivityHc, magnetic-materials.AnisotropyField, magnetic-materials.MassMagnetizationUnit, magnetic-materials.CoercivityHcExternal, magnetic-materials.InternalSusceptibility, magnetic-materials.SpontaneousMagnetization, magnetic-materials.UniaxialAnisotropyConstant, magnetic-materials.DemagnetizingField, magnetic-materials.SaturationMagnetization, magnetic-materials.BinderCumulant]